# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library, with all entities referenced by their Croissant schema `@id` fields for full reproducibility and auditability.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load croissant dataset metadata and explore available entities.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata

print(f"Dataset Name: {md.name}")
print(f"Description: {md.description}\n")
print(f"Citation: {md.citeAs}")
print(f"License: {md.license}")
print(f"Version: {md.version}")

## 2. Data Overview
List available record sets, field `@id`s, and their key properties as defined in the Croissant schema. All entities are referenced by their `@id` field.

In [ ]:
# Discover available record sets and their fields

record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description or '(No description)'}")
    print(f"  Fields:")
    for field in rs.fields:
        field_type = getattr(field, 'data_type', '(unknown)')
        print(f"    - Field @id: {field.id}; Name: {field.name}; Type: {field_type}")

## 3. Data Extraction
Extract data from the primary record set defined by its `@id`. (Use the correct `@id`s discovered in the previous block.)

In [ ]:
# We'll extract all records from the main RecordSet for analysis.
# Replace with discovered @id for the main data table.

# --- AUTO-DETECT the main RecordSet suitable for tabular analysis ---
# Use the first RecordSet for demonstration (you may select others as needed)

main_record_set = record_sets[0]  # Use the first record set
main_record_set_id = main_record_set.id  # This is the @id
print(f"Loading records from RecordSet @id={main_record_set_id}")

# Get all field @ids from this record set
field_ids = [f.id for f in main_record_set.fields]
print("Fields in this RecordSet:", field_ids)

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records with columns:")
pprint(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform filtering, normalization, and grouping using appropriate field `@id`s. We'll select a numeric field and a grouping field from those listed in the record set.

In [ ]:
# Choose a numeric field for filtering/analysis
# We'll auto-select the first field of numeric type if possible
numeric_field_id = None
for f in main_record_set.fields:
    dt = str(getattr(f, 'data_type', '')).lower()
    if dt in ['float', 'integer', 'number', 'double']:
        numeric_field_id = f.id
        break

if not numeric_field_id:
    print("No numeric field found. Please select one manually.")
else:
    print(f"Numeric field selected (@id): {numeric_field_id}")

    # Filtering
    threshold = 10
    df_numeric = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df_numeric > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    display_cols = [numeric_field_id]
    print(filtered_df[display_cols].head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (df_numeric - df_numeric.mean()) / df_numeric.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a categorical field
    group_field_id = None
    for f in main_record_set.fields:
        dt = str(getattr(f, 'data_type', '')).lower()
        if dt in ['text', 'string'] and f.id != numeric_field_id:
            group_field_id = f.id
            break
    
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and the group means by category if a grouping field was found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].astype(float), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load and inspect a Croissant-structured FAIR dataset using the `mlcroissant` library
- Programmatically list and reference record sets, fields, and numeric/categorical columns by their `@id`s
- Perform basic tabular filtering, normalization, and group aggregation using only Croissant.
- Visualize distributions in a reproducible script referencing all data elements by schema `@id`

This pattern enables reproducible, transparent FAIR data workflows for mass spectrometry and any other Croissant-based data resource.
